In [22]:
import os
import re
import sys
import json
import torch
import numpy as np
import subprocess
import unicodedata
from pathlib import Path
from datetime import timedelta
from PIL import Image
import soundfile as sf
import scipy.io.wavfile
from scipy.io import wavfile
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import open_clip
import time

from pocket_tts import TTSModel

from sqlalchemy.orm import Session
from sqlalchemy import text
from transformers import BlipProcessor, BlipForConditionalGeneration
import boto3
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor

In [2]:
root = Path.cwd()
while not (root / "src").exists():
    root = root.parent

sys.path.append(str(root))
from src.db.session import engine
from src.models import Pharaoh

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

SCENE_SIM_THRESHOLD = 0.65
IMAGE_SIM_WEIGHT = 0.3
DESC_SIM_WEIGHT = 0.7
MAX_IMAGE_DURATION = 7

device

'cuda'

In [4]:
import open_clip

model_path = "C:\\Users\\lidia\\Downloads\\open_clip_pytorch_model.bin"

model, preprocess, tokenizer = open_clip.create_model_and_transforms(
    "ViT-H-14",
    pretrained=model_path
)

model = model.to(device)
model.eval()


CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-31): 32 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1280, out_features=1280, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1280, out_features=5120, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=5120, out_features=1280, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1280,), eps=1e-05, elementwi

In [ ]:
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

c:\Users\lidia\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lidia\.cache\huggingface\hub\models--Salesforce--blip-image-captioning-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
The image processor of type `BlipImageProcessor` is now loaded as a fast pro

In [ ]:
client = OpenAI(
    api_key="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",
    base_url="https://api.groq.com/openai/v1"
)

In [6]:
script_path = r'D:\1Graduation project\ECHO\data\video_generation\outputs\pharaohs_scripts\Horus (God).txt'
with open(script_path, 'r', encoding='utf-8') as file:
    script = file.read()
paragraphs = [p.strip() for p in re.split(r'\n\s*\n', script) if p.strip()]
paragraphs

["Horus, one of Egypt's oldest gods, has been revered for thousands of years. Early inscriptions show him with outstretched wings as protector to the nation’s rulers and in the serekh—a falcon on a perch—symbolizing his role from around 2850 B.C.E.",
 "This protective image was repeated throughout Egyptian history, famously seen in Khafre's statue where Horus stands guard. In later times, both Horus and Set were depicted as bringing the double crowns of Upper and Lower Egypt to pharaohs like Khafre.",
 "Horus’s cult centers thrived at temples such as Edfu, with its temple becoming a major site for worship from late predynastic times onward. The goddess Wadjet, protector of Lower Egypt, kept watch over Horus's divine family during their early years in the Nile delta.",
 'In battles and myths, Horus lost his left eye—the moon—during a fight with Set but was healed by Thoth, giving rise to the Wedjat (Eye of Horus) symbol still recognized today. This powerful myth explains why we see diff

In [7]:
def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

sentences = split_sentences(script)
sentences

["Horus, one of Egypt's oldest gods, has been revered for thousands of years.",
 'Early inscriptions show him with outstretched wings as protector to the nation’s rulers and in the serekh—a falcon on a perch—symbolizing his role from around 2850 B.C.E.',
 "This protective image was repeated throughout Egyptian history, famously seen in Khafre's statue where Horus stands guard.",
 'In later times, both Horus and Set were depicted as bringing the double crowns of Upper and Lower Egypt to pharaohs like Khafre.',
 'Horus’s cult centers thrived at temples such as Edfu, with its temple becoming a major site for worship from late predynastic times onward.',
 "The goddess Wadjet, protector of Lower Egypt, kept watch over Horus's divine family during their early years in the Nile delta.",
 'In battles and myths, Horus lost his left eye—the moon—during a fight with Set but was healed by Thoth, giving rise to the Wedjat (Eye of Horus) symbol still recognized today.',
 'This powerful myth explains

In [8]:
sentence_model = SentenceTransformer("all-mpnet-base-v2")
sentence_embeddings = sentence_model.encode(sentences, normalize_embeddings=True)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 710.20it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
def cosine(a,b):
    return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))

scenes = []
current_scene = [sentences[0]]

for i in range(len(sentences)-1):

    sim = cosine(sentence_embeddings[i], sentence_embeddings[i+1])

    if sim < SCENE_SIM_THRESHOLD:
        scenes.append(" ".join(current_scene))
        current_scene = []

    current_scene.append(sentences[i+1])

if current_scene:
    scenes.append(" ".join(current_scene))

print("Detected scenes:", len(scenes))

Detected scenes: 9


In [10]:
tts_model = TTSModel.load_model()
voice_state = tts_model.get_state_for_audio_prompt("alba")

Path("tts_Outputs").mkdir(exist_ok=True)

durations = []
scene_audio_files = []

for i, scene in enumerate(scenes):

    audio = tts_model.generate_audio(voice_state, scene)

    output = f"tts_Outputs/scene_{i}.wav"

    scipy.io.wavfile.write(output, tts_model.sample_rate, audio.numpy())

    Fs,data = wavfile.read(output)

    duration = len(data)/float(Fs)

    durations.append(duration)
    scene_audio_files.append(output)

    print("Scene",i,"duration",duration)

Scene 0 duration 5.6


Scene 1 duration 12.08
Scene 2 duration 6.32
Scene 3 duration 7.44
Scene 4 duration 7.76
Scene 5 duration 6.32
Scene 6 duration 11.84
Scene 7 duration 4.24
Scene 8 duration 5.28


In [11]:
audio_data=[]
samplerate=None
for f in scene_audio_files:
    data,sr = sf.read(f)
    if samplerate is None:
        samplerate=sr
    audio_data.append(data)

combined = np.concatenate(audio_data, axis=0)
sf.write("tts_Outputs/final_audio.wav", combined, samplerate)

In [12]:
from sqlalchemy.orm import Session
from sqlalchemy import text

name = Path(script_path).stem

with Session(engine) as session:
    result = session.execute(
        text("""
            SELECT 
                pi.id,
                pi.image_path,
                pi.image_embedding,
                pi.image_description
            FROM pharaohs_images pi
            JOIN pharaohs p ON pi.pharaoh_id = p.id
            WHERE p.name = :name
        """),
        {"name": name}
    )

    images_data=result.fetchall()

print(f"\nFound {len(images_data)} images for {name}")


Found 18 images for Horus (God)


In [13]:
clip_tokenizer = open_clip.get_tokenizer("ViT-H-14")
processed_images=[]
for row in images_data:
    image_id,image_path,image_embedding,image_description=row
    if isinstance(image_embedding,str):
        image_embedding=json.loads(image_embedding)

    image_embedding=np.array(image_embedding)
    image_embedding=image_embedding/np.linalg.norm(image_embedding)

    tokens=clip_tokenizer([image_description]).to(device)

    with torch.no_grad():
        desc_emb=model.encode_text(tokens)
        desc_emb/=desc_emb.norm(dim=-1,keepdim=True)

    desc_emb=desc_emb.cpu().numpy()[0]

    processed_images.append({
        "id": image_id,
        "path": image_path,
        "img_emb": image_embedding,
        "desc_emb": desc_emb
    })

In [14]:
def generate_visual_prompt(scene_text):

    prompt = f"""
You are an expert visual prompt generator for an Ancient Egypt image retrieval system.

Convert the narration into a SHORT visual description of what an image should show, that will help retrieve the best matching historical image from an Ancient Egypt dataset.


The image database contains:
- Pharaoh statues
- Egyptian gods and goddesses
- Royal family members (father, son, wife, daughter)
- Temple relief carvings
- Archaeological artifacts
- Monuments and temples

Guidelines:
1. Identify the MAIN ENTITY (pharaoh, god, goddess, royal family).
2. Include the NAME if present.
3. Include VISUAL ELEMENTS such as: statue, temple relief, wall carving, hieroglyphs, monument, artifact.
4. Include LOCATION if mentioned (temple name, city, monument).
5. Prefer descriptions that look like museum or archaeological images.
6. Use concise keyword-style phrases, not storytelling.

Output a single visual description.

Narration:
{scene_text}

Visual prompt:
"""

    time.sleep(0.2)
    response = client.chat.completions.create(
        model="qwen/qwen3-32b",
        messages=[
            {"role": "system", "content": "You create visual prompts for documentary scenes."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )

    return response.choices[0].message.content.strip()

In [ ]:
used_image_ids_global = set()
TOP_K = 5

def select_sentence_image(scene_text, scene_dur, processed_images, max_image_duration=MAX_IMAGE_DURATION):
    visual_prompt = generate_visual_prompt(scene_text)
    print("\nGenerated Prompt:")
    print(visual_prompt)
    
    prompt_tokens = clip_tokenizer([visual_prompt]).to(device)
    with torch.no_grad():
        prompt_emb = model.encode_text(prompt_tokens)
        prompt_emb /= prompt_emb.norm(dim=-1, keepdim=True)
    prompt_emb = prompt_emb.cpu().numpy()[0]

    scene_tokens = clip_tokenizer([scene_text]).to(device)
    with torch.no_grad():
        scene_emb = model.encode_text(scene_tokens)
        scene_emb /= scene_emb.norm(dim=-1, keepdim=True)
    scene_emb = scene_emb.cpu().numpy()[0]

    # Rank images by similarity
    ranked = []
    for img in processed_images:
        image_sim = cosine(prompt_emb, img["img_emb"])
        desc_sim = cosine(scene_emb, img["desc_emb"])
        score = IMAGE_SIM_WEIGHT * image_sim + DESC_SIM_WEIGHT * desc_sim
        ranked.append((score, img["id"], img["path"]))
        
    ranked.sort(reverse=True, key=lambda x: x[0])

    for score, img_id, img_path in ranked:
        if img_id not in used_image_ids_global:
            used_image_ids_global.add(img_id)
            return [img_path], [scene_dur]

    return [ranked[0][2]], [scene_dur]



In [ ]:
expanded_images = []
expanded_durations = []
sentence_scripts = []

for scene_text, scene_dur in zip(scenes, durations):
        img_path, img_dur = select_sentence_image(scene_text, scene_dur, processed_images, MAX_IMAGE_DURATION)
        for img_path, img_dur in zip(img_path, img_dur):
            expanded_images.append(img_path)
            expanded_durations.append( scene_dur)
            sentence_scripts.append(scene_text)
            filename = Path(img_path).name
            print(f"Image: {filename} | \nScript: {scene_text}\n")

print("Total images selected:", len(expanded_images))
print("Total durations:", expanded_durations)


Generated Prompt:
<think>
Okay, let's tackle this query. The user wants a visual prompt for an image retrieval system focused on Ancient Egypt. The narration given is about Horus, one of Egypt's oldest gods, revered for thousands of years.

First, I need to identify the main entity. The narration clearly mentions Horus, who is a god. So the main entity is Horus, a god. Next, the guidelines say to include the name if present, which it is. 

Now, visual elements. Since Horus is a god, common visual representations would be statues or temple reliefs. The database includes Pharaoh statues, Egyptian gods, temple reliefs, etc. So possible elements could be a statue of Horus or a temple relief depicting him. Also, maybe hieroglyphs associated with him. 

Location isn't specified in the narration, so I can omit that unless there's a common place Horus is depicted. But since the user says to include location if mentioned, and there's none here, leave it out. 

Prefer museum or archaeological i

In [17]:
load_dotenv()

ACCOUNT_ID = os.getenv("R2_ACCOUNT_ID")
ACCESS_KEY = os.getenv("R2_ACCESS_KEY")
SECRET_KEY = os.getenv("R2_SECRET_KEY")
BUCKET_NAME = os.getenv("R2_BUCKET_NAME")

session = boto3.session.Session()

client = session.client(
    "s3",
    region_name="auto",
    endpoint_url=f"https://{ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
)

local_frames_dir = Path("temp_frames")
local_frames_dir.mkdir(exist_ok=True)

def download_image(idx_key):
    idx, image_key = idx_key
    local_file = local_frames_dir / f"{idx:04d}.jpg"
    client.download_file(BUCKET_NAME, image_key, str(local_file))
    return str(local_file)

with ThreadPoolExecutor(max_workers=8) as exe:
    image_files = list(exe.map(download_image, enumerate(expanded_images)))

In [18]:
print(expanded_images)
print(expanded_durations)

['data/video_generation/raw/pharaohs_images/Horus (God)/Horus God.jpg', 'data/video_generation/raw/pharaohs_images/Horus (God)/Horus Relief.jpg', 'data/video_generation/raw/pharaohs_images/Horus (God)/Horus Relief 2.jpg', 'data/video_generation/raw/pharaohs_images/Horus (God)/Eye of Horus.jpg', 'data/video_generation/raw/pharaohs_images/Horus (God)/Horus stela.jpg', 'data/video_generation/raw/pharaohs_images/Horus (God)/Horus, Osiris and Isis.jpg', 'data/video_generation/raw/pharaohs_images/Horus (God)/Horus Conception.jpg', 'data/video_generation/raw/pharaohs_images/Horus (God)/Statue 1.jpg', 'data/video_generation/raw/pharaohs_images/Horus (God)/Horus.jpg']
[5.36, 13.36, 6.8, 8.64, 8.56, 6.16, 12.16, 4.32, 6.4]


In [19]:
image_files = list(local_frames_dir.glob("*.jpg")) + list(local_frames_dir.glob("*.jpeg"))

In [ ]:
import re
import unicodedata
from datetime import timedelta

MAX_CHARS_PER_LINE = 42
MAX_LINES = 2
MIN_DURATION = 1.0


def normalize_text(text):

    text = unicodedata.normalize("NFKC", text)

    replacements = {
        "’": "'",
        "‘": "'",
        "‚": ",",
        "‛": "'",

        "“": '"',
        "”": '"',
        "„": '"',

        "—": "-",
        "–": "-",
        "―": "-",

        "…": "...",

        "\u00A0": " ",
        "\u200B": "",
        "\u200C": "",
        "\u200D": "",
        "\uFEFF": "",
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    text = "".join(
        ch for ch in text
        if unicodedata.category(ch)[0] != "C"
    )

    return text


def format_timestamp(seconds):

    td = timedelta(seconds=seconds)
    total_seconds = int(td.total_seconds())
    millis = int((seconds - total_seconds) * 1000)

    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    secs = total_seconds % 60

    return f"{hours:02}:{minutes:02}:{secs:02},{millis:03}"


def split_into_sentences(text):
    return re.split(r'(?<=[.!?]) +', text.strip())


def split_long_text(text, max_chars=MAX_CHARS_PER_LINE):

    words = text.split()

    lines = []
    current_line = ""

    for word in words:

        if len(current_line) + len(word) + 1 <= max_chars:
            current_line += (" " if current_line else "") + word

        else:
            lines.append(current_line)
            current_line = word

    if current_line:
        lines.append(current_line)

    blocks = []

    for i in range(0, len(lines), MAX_LINES):

        block = "\n".join(lines[i:i + MAX_LINES])
        blocks.append(block)

    return blocks


def generate_srt(text_blocks, durations, output_path):

    assert len(text_blocks) == len(durations), "Text blocks and durations must match"

    current_time = 0.0
    subtitle_index = 1
    srt_blocks = []

    for text, duration in zip(text_blocks, durations):

        text = normalize_text(text)

        sentences = split_into_sentences(text)

        chunks = []

        for sentence in sentences:
            chunks.extend(split_long_text(sentence))

        if len(chunks) == 0:
            continue

        total_chars = sum(len(chunk.replace("\n", "")) for chunk in chunks)

        if total_chars == 0:
            continue

        for chunk in chunks:

            chunk_chars = len(chunk.replace("\n", ""))

            chunk_duration = max(
                MIN_DURATION,
                (chunk_chars / total_chars) * duration
            )

            start_time = current_time
            end_time = current_time + chunk_duration

            srt_blocks.append(
                f"{subtitle_index}\n"
                f"{format_timestamp(start_time)} --> {format_timestamp(end_time)}\n"
                f"{chunk}\n\n"
            )

            current_time = end_time
            subtitle_index += 1

    with open(output_path, "w", encoding="utf-8-sig") as f:
        f.writelines(srt_blocks)

    print("SRT file saved:", output_path)

In [21]:
generate_srt(
    scenes,
    durations,
    "tts_Outputs/output_subtitles.srt"
)

SRT file saved: tts_Outputs/output_subtitles.srt


In [ ]:
def run_ffmpeg(cmd):
    cmd = [str(c) for c in cmd]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

def create_kenburns_clip(
    image_path,
    duration,
    output_path,
    fps=30,
    target_w=1920,
    target_h=1080
):
    total_frames = int(duration * fps)

    with Image.open(image_path) as img:
        img_w, img_h = img.size

    scale_factor = max(target_w / img_w, target_h / img_h)

    scaled_w = int(img_w * scale_factor)
    scaled_h = int(img_h * scale_factor)

    scaled_w = scaled_w if scaled_w % 2 == 0 else scaled_w + 1
    scaled_h = scaled_h if scaled_h % 2 == 0 else scaled_h + 1

    max_x = scaled_w - target_w
    max_y = scaled_h - target_h

    threshold = 40  

    ease = f"(0.5-0.5*cos(PI*n/{total_frames}))"
    
    if (img_w / img_h) > (target_w / target_h):
        if max_x > threshold:
            x_expr = f"floor({max_x}*{ease})"
            y_expr = f"{max_y}/2"
            zoom_filter = ""
        else:
            zoom_filter = f",scale=iw*1.05:ih*1.05"
            x_expr = f"(iw-{target_w})/2"
            y_expr = f"(ih-{target_h})/2"
    else:
        if max_y > threshold:
            x_expr = f"{max_x}/2"
            y_expr = f"floor({max_y}*{ease})"
            zoom_filter = ""
        else:
            zoom_filter = f",scale=iw*1.05:ih*1.05"
            x_expr = f"(iw-{target_w})/2"
            y_expr = f"(ih-{target_h})/2"

    vf = (
        f"scale={scaled_w}:{scaled_h}"
        f"{zoom_filter},"
        f"crop={target_w}:{target_h}:"
        f"x='{x_expr}':"
        f"y='{y_expr}'"
    )

    cmd = [
        "ffmpeg",
        "-y",
        "-loop", "1",
        "-framerate", str(fps),
        "-t", str(duration),
        "-i", image_path,
        "-vf", vf,
        "-frames:v", str(total_frames),
        "-c:v", "libx264",
        "-preset", "fast",
        "-pix_fmt", "yuv420p",
        "-vsync", "cfr",
        output_path
    ]

    run_ffmpeg(cmd)
    


In [23]:
def generate_all_clips(image_files, expanded_durations, temp_dir="temp_clips"):
    Path(temp_dir).mkdir(exist_ok=True)
    outputs = []
    for i, (img, dur) in enumerate(zip(image_files, expanded_durations)):
        out = f"{temp_dir}/clip_{i}.mp4"
        create_kenburns_clip(img, dur, out)
        outputs.append(out)
    return outputs


def concatenate_clips(clips, output_path):
    list_file = "tts_Outputs/concat_list.txt"

    with open(list_file, "w") as f:
        for clip in clips:
            f.write(f"file '{os.path.abspath(clip)}'\n")

    cmd = [
        "ffmpeg",
        "-y",
        "-f", "concat",
        "-safe", "0",
        "-i", list_file,
        "-c:v", "libx264",
        "-preset", "fast",
        #"-c:a", "aac",
        output_path
    ]

    run_ffmpeg(cmd)

In [24]:
def add_audio(video_path, audio_path, output_path):
    cmd = [
        "ffmpeg",
        "-y",
        "-i", video_path,
        "-i", audio_path,
        "-c:v", "copy",
        "-c:a", "aac",
        output_path
    ]

    run_ffmpeg(cmd)

def add_subtitles(video_path, srt_path, output_path):
    cmd = [
        "ffmpeg",
        "-y",
        "-i", video_path,
        "-vf", f"subtitles={srt_path}",
        "-c:v", "libx264",
        "-preset", "fast",
        "-c:a", "copy",
        output_path
    ]

    run_ffmpeg(cmd)

def cleanup_files():
    temp_dir = "temp_clips"
    if os.path.exists(temp_dir):
        for f in os.listdir(temp_dir):
            os.remove(os.path.join(temp_dir, f))
        os.rmdir(temp_dir)

    temp_dir = "temp_frames"
    if os.path.exists(temp_dir):
        for f in os.listdir(temp_dir):
            os.remove(os.path.join(temp_dir, f))
        os.rmdir(temp_dir)

    for f in os.listdir('tts_Outputs'):
        if f.startswith("combined") or f.startswith("with_audio") or f.startswith("concat_list") or f.startswith("output_subtitles") or f.endswith(".wav"):
            os.remove(os.path.join('tts_Outputs', f))

In [25]:
print(image_files)
import subprocess

subprocess.run(["ffmpeg", "-version"], check=True)

[WindowsPath('temp_frames/0000.jpg'), WindowsPath('temp_frames/0001.jpg'), WindowsPath('temp_frames/0002.jpg'), WindowsPath('temp_frames/0003.jpg'), WindowsPath('temp_frames/0004.jpg'), WindowsPath('temp_frames/0005.jpg'), WindowsPath('temp_frames/0006.jpg'), WindowsPath('temp_frames/0007.jpg'), WindowsPath('temp_frames/0008.jpg')]


CompletedProcess(args=['ffmpeg', '-version'], returncode=0)

In [26]:
clips = generate_all_clips(image_files, expanded_durations)
concatenated_video = "tts_Outputs/combined.mp4"
concatenate_clips(clips, concatenated_video)

video_with_audio = "tts_Outputs/with_audio.mp4"
add_audio(concatenated_video, "tts_Outputs/final_audio.wav", video_with_audio)

generate_srt(scenes, durations, "tts_Outputs/output_subtitles.srt")

final_output = "tts_Outputs/image_oriental1_final_video.mp4"
add_subtitles(video_with_audio, "tts_Outputs/output_subtitles.srt", final_output)

cleanup_files()
print("Video generation finished!")
print("Final video:", final_output)

Running: ffmpeg -y -loop 1 -framerate 30 -t 5.36 -i temp_frames\0000.jpg -vf scale=1920:3082,crop=1920:1080:x='0/2':y='floor(2002*(0.5-0.5*cos(PI*n/160)))' -frames:v 160 -c:v libx264 -preset fast -pix_fmt yuv420p -vsync cfr temp_clips/clip_0.mp4
Running: ffmpeg -y -loop 1 -framerate 30 -t 13.36 -i temp_frames\0001.jpg -vf scale=1920:2824,crop=1920:1080:x='0/2':y='floor(1744*(0.5-0.5*cos(PI*n/400)))' -frames:v 400 -c:v libx264 -preset fast -pix_fmt yuv420p -vsync cfr temp_clips/clip_1.mp4
Running: ffmpeg -y -loop 1 -framerate 30 -t 6.8 -i temp_frames\0002.jpg -vf scale=1920:2320,crop=1920:1080:x='0/2':y='floor(1240*(0.5-0.5*cos(PI*n/204)))' -frames:v 204 -c:v libx264 -preset fast -pix_fmt yuv420p -vsync cfr temp_clips/clip_2.mp4
Running: ffmpeg -y -loop 1 -framerate 30 -t 8.64 -i temp_frames\0003.jpg -vf scale=1920:1536,crop=1920:1080:x='0/2':y='floor(456*(0.5-0.5*cos(PI*n/259)))' -frames:v 259 -c:v libx264 -preset fast -pix_fmt yuv420p -vsync cfr temp_clips/clip_3.mp4
Running: ffmpeg -